In [0]:
#Install kagglehub library
%pip install kagglehub

In [0]:
# Import library
import kagglehub
import os
from pyspark.sql import SparkSession

In [0]:
# Find the path of the dataset
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)


In [0]:
#Display files
files = dbutils.fs.ls(f"file:{path}")
display(files)


In [0]:
spark = SparkSession.builder \
    .appName("Olist Bronze Ingestion") \
    .enableHiveSupport() \
    .getOrCreate()

# Bronze database in Hive metastore
bronze_db = "bronze_olist"
spark.sql(f"DROP DATABASE IF EXISTS {bronze_db} CASCADE")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")



In [0]:
#Check files in the directory
for file in files:
    df = spark.read.option("header", True).csv(file.path)  # read CSV
    display(file.name)
    display(df.limit(5))
    


In [0]:
#Check file names
for file in files: 
    print(file.name)

In [0]:
for file in files:
    df = spark.read.option("header", True).csv(file.path)  # read CSV
    #Clean the name of csv
    name = os.path.splitext(file.name)[0].replace('_dataset','').replace('_translation','').replace('olist_','')
    # Save as Hive table in bronze database
    df.write.mode("overwrite").saveAsTable(f"{bronze_db}.{name}")
    
    print(f"{name} ingested to bronze layer")
print(f'{len(files)} files ingested to {bronze_db}')